In [12]:
import random
num_locations = 4
locations = ['A', 'B', 'C', 'D']
start_location = 'A'
cost_matrix = [
    [0, 12, 18, 10],
    [12, 0, 20, 15],
    [18, 20, 0, 25],
    [10, 15, 25, 0]
]
idx_to_loc = {i: loc for i, loc in enumerate(locations)}

In [13]:
def create_initial_population(pop_size, num_locs):
    population = []
    locations_to_visit = list(range(1, num_locs))
    for i in range(pop_size):
        chromosome = locations_to_visit.copy()
        random.shuffle(chromosome)
        population.append(chromosome)
    return population
sample_population = create_initial_population(pop_size=5, num_locs=4)
print("sample initial population:", sample_population)

sample initial population: [[1, 2, 3], [2, 3, 1], [2, 3, 1], [2, 3, 1], [2, 1, 3]]


In [14]:
def calculate_route_cost(chromosome, cost_mat):
    total_cost = 0
    current_loc = 0 
    for next_loc in chromosome:
        total_cost += cost_mat[current_loc][next_loc]
        current_loc = next_loc
    total_cost += cost_mat[current_loc][0] 
    return total_cost

def calculate_fitness(chromosome, cost_mat):
    cost = calculate_route_cost(chromosome, cost_mat)
    if cost == 0:
        return float('inf')
    return 1.0 / cost
example_chromosome = [1, 2, 3]
print(f"Cost for {example_chromosome}: {calculate_route_cost(example_chromosome, cost_matrix)}")

Cost for [1, 2, 3]: 67


In [15]:
def select_parents(population, fitnesses, tournament_size=3):
    parents = []
    
    for i in range(2):
        tournament_indices = random.sample(range(len(population)), min(tournament_size, len(population)))
        
        best_idx = tournament_indices[0]
        best_fitness = fitnesses[best_idx]
        
        for idx in tournament_indices[1:]:
            if fitnesses[idx] > best_fitness:
                best_fitness = fitnesses[idx]
                best_idx = idx
                
        parents.append(population[best_idx])
        
    return parents[0], parents[1]

In [16]:
def crossover(parent1, parent2):
    size = len(parent1)
    if size <= 1:
        return parent1.copy()
    start, end = sorted(random.sample(range(size), 2))
    offspring = [None] * size
    offspring[start:end+1] = parent1[start:end+1]
    ptr2 = 0
    for i in range(size):
        if offspring[i] is None:
            while parent2[ptr2] in offspring:
                ptr2 += 1
            offspring[i] = parent2[ptr2]
            
    return offspring

In [17]:
def mutate(chromosome, mutation_rate=0.1):
    mutated = chromosome.copy()
    if random.random() < mutation_rate and len(chromosome) >= 2:
        idx1, idx2 = random.sample(range(len(chromosome)), 2)
        mutated[idx1], mutated[idx2] = mutated[idx2], mutated[idx1]
        
    return mutated

In [19]:
def genetic_algorithm(cost_mat, num_locs, pop_size=10, generations=20, mutation_rate=0.1):
    population = create_initial_population(pop_size, num_locs)
    
    best_overall_route = None
    best_overall_cost = float('inf')
    
    for generation in range(generations):
        fitnesses = [calculate_fitness(chrom, cost_mat) for chrom in population]
        for chrom in population:
            cost = calculate_route_cost(chrom, cost_mat)
            if cost < best_overall_cost:
                best_overall_cost = cost
                best_overall_route = chrom.copy()
                
        new_population = []
        new_population.append(best_overall_route)
        while len(new_population) < pop_size:
            parent1, parent2 = select_parents(population, fitnesses)
            offspring = crossover(parent1, parent2)
            offspring = mutate(offspring, mutation_rate)
            new_population.append(offspring)
        population = new_population
        
    return best_overall_route, best_overall_cost
best_route_indices, best_cost = genetic_algorithm(
    cost_matrix, 
    num_locations, 
    pop_size=2, 
    generations=5, 
    mutation_rate=0.1
)
route_names = ['A'] + [idx_to_loc[idx] for idx in best_route_indices] + ['A']
route_str = "->".join(route_names)

print("final result")
print(f"best route: {route_str}")
print(f"minimum total cost: {best_cost}")

final result
best route: A->D->C->B->A
minimum total cost: 67
